# Delivery Time Prediction

**Tabular Regression Project**

Models: CatBoost · LightGBM · XGBoost
Baselines: LazyPredict · FLAML AutoML
Target: `delivery_minutes`

## 2 · Project Overview

We predict **delivery time** (minutes) for food/package deliveries based on distance, order time, weather, vehicle type, restaurant quality, order size, and traffic conditions.

## 3 · Learning Objectives

By the end of this notebook you will be able to:
1. Perform EDA on a tabular regression dataset.
2. Encode categorical features for tree-based and linear models.
3. Build a baseline Linear Regression model.
4. Use LazyPredict for rapid model benchmarking.
5. Run FLAML AutoML for automated model selection.
6. Train CatBoost, LightGBM, XGBoost with GPU→CPU fallback.
7. Evaluate with RMSE, MAE, R², and residual analysis.

## 4 · Problem Statement

Given delivery distance, order hour, weather, vehicle type, restaurant rating, number of items, and traffic level, predict the total delivery time in minutes.

## 5 · Why This Project Matters

- **Delivery time estimation** directly impacts customer satisfaction.
- Accurate ETAs reduce customer complaints and improve retention.
- Route and vehicle optimization depend on time predictions.
- Demonstrates regression with additive time components.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Rows** | 8,000 |
| **Features** | distance_km, order_hour, is_peak_hour, day_of_week, is_weekend, weather, vehicle_type, restaurant_rating, num_items, city_traffic |
| **Target** | delivery_minutes (continuous) |
| **Range** | ~5 – 120 minutes |

## 7 · Dataset Source and License Notes

- **Source**: Synthetic dataset generated for educational purposes.
- **License**: Public / educational use. No PII.
- **Limitations**: Simplified patterns — real-world data would have more features and noise.

## 8 · Environment Setup

Install any missing packages. GPU is auto-detected with CPU fallback.

In [ ]:
import subprocess, sys

def _install(pkg):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["catboost", "lightgbm", "xgboost", "flaml", "lazypredict"]:
    _install(pkg)

print("All packages ready.")

## 9 · Imports

In [ ]:
import os, json, time, warnings, random, gc
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, root_mean_squared_error

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Imports complete.")

## 10 · Configuration / Constants

In [ ]:
TARGET = "delivery_minutes"
SAVE_DIR = os.path.dirname(os.path.abspath("__file__"))
print(f"Target: {TARGET}")
print(f"Save dir: {SAVE_DIR}")

## 11 · Dataset Download or Loading

Synthetic dataset: 8,000 food/package delivery records with route and order features.

In [ ]:
np.random.seed(SEED)
n = 8000
distance_km = np.round(np.random.lognormal(1.5, 0.6, n).clip(0.5, 30), 1)
order_hour = np.random.randint(8, 23, n)
is_peak_hour = ((order_hour >= 11) & (order_hour <= 13) | (order_hour >= 18) & (order_hour <= 20)).astype(int)
day_of_week = np.random.randint(0, 7, n)
is_weekend = (day_of_week >= 5).astype(int)
weather = np.random.choice(["Clear", "Rain", "Snow"], n, p=[0.65, 0.25, 0.1])
weather_delay = np.where(weather == "Snow", 12, np.where(weather == "Rain", 5, 0))
vehicle_type = np.random.choice(["Bicycle", "Motorcycle", "Car"], n, p=[0.3, 0.45, 0.25])
vehicle_speed = np.where(vehicle_type == "Bicycle", 0.8, np.where(vehicle_type == "Motorcycle", 1.0, 1.1))
restaurant_rating = np.round(np.random.uniform(2.5, 5.0, n), 1)
num_items = np.random.randint(1, 10, n)
city_traffic = np.random.choice(["Low", "Medium", "High"], n, p=[0.3, 0.45, 0.25])
traffic_mult = np.where(city_traffic == "High", 1.4, np.where(city_traffic == "Medium", 1.15, 1.0))

# Delivery time model
prep_time = 8 + 2 * num_items + (5 - restaurant_rating) * 2
travel_time = (distance_km * 3.5) / vehicle_speed * traffic_mult
peak_delay = 8 * is_peak_hour

delivery_minutes = prep_time + travel_time + weather_delay + peak_delay
delivery_minutes = np.round(delivery_minutes + np.random.normal(0, 5, n), 1).clip(5, 120)

df = pd.DataFrame({"distance_km": distance_km, "order_hour": order_hour,
                    "is_peak_hour": is_peak_hour, "day_of_week": day_of_week,
                    "is_weekend": is_weekend, "weather": weather,
                    "vehicle_type": vehicle_type, "restaurant_rating": restaurant_rating,
                    "num_items": num_items, "city_traffic": city_traffic,
                    "delivery_minutes": delivery_minutes})
print(f"Dataset shape: {df.shape}")
print(f"Target stats:\n{df['delivery_minutes'].describe()}")
df.head()

## 12 · Data Validation Checks

Verify the dataset is complete and correctly structured.

In [ ]:
print("=" * 50)
print("DATA VALIDATION")
print("=" * 50)
print(f"\nShape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"\nDtypes:\n{df.dtypes}")
assert TARGET in df.columns, f"Target '{TARGET}' not found!"
print(f"\nTarget '{TARGET}' confirmed.")
print(f"\nTarget stats:\n{df[TARGET].describe()}")

## 13 · Exploratory Data Analysis

Explore feature distributions, correlations, and relationships.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

axes[0][0].scatter(df["distance_km"], df[TARGET], alpha=0.2, s=10)
axes[0][0].set_xlabel("Distance (km)"); axes[0][0].set_ylabel("Delivery (min)")
axes[0][0].set_title("Distance vs Delivery Time")

axes[0][1].scatter(df["num_items"], df[TARGET], alpha=0.2, s=10)
axes[0][1].set_xlabel("# Items"); axes[0][1].set_ylabel("Delivery (min)")
axes[0][1].set_title("Items vs Delivery Time")

df.boxplot(column=TARGET, by="weather", ax=axes[0][2])
axes[0][2].set_title("Delivery by Weather")

df.boxplot(column=TARGET, by="vehicle_type", ax=axes[1][0])
axes[1][0].set_title("Delivery by Vehicle")

df.boxplot(column=TARGET, by="city_traffic", ax=axes[1][1])
axes[1][1].set_title("Delivery by Traffic")

corr = df.select_dtypes(include="number").corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=axes[1][2], square=True, cbar_kws={"shrink":0.8})
axes[1][2].set_title("Correlation")

plt.tight_layout()
plt.show()

## 14 · Target Analysis

Examine the distribution of `delivery_minutes`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df[TARGET], bins=30, color="steelblue", edgecolor="black", alpha=0.8)
axes[0].set_title(f"Distribution of {TARGET}")
axes[0].set_xlabel("Delivery Time (minutes)")

df.groupby("order_hour")[TARGET].mean().plot(ax=axes[1], marker="o", color="steelblue")
axes[1].set_title("Avg Delivery Time by Hour")
axes[1].set_xlabel("Order Hour"); axes[1].set_ylabel("Minutes")

plt.tight_layout()
plt.show()
print(f"Range: {df[TARGET].min():.1f} to {df[TARGET].max():.1f} min")
print(f"Mean: {df[TARGET].mean():.1f}, Median: {df[TARGET].median():.1f}")
print(f"Peak hour premium: {df[df['is_peak_hour']==1][TARGET].mean() - df[df['is_peak_hour']==0][TARGET].mean():.1f} min")

## 15 · Train/Validation/Test Split Strategy

80/20 train/test split.

In [ ]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

# Encode categorical features
cat_cols = X.select_dtypes(include="object").columns.tolist()
if cat_cols:
    oe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    X[cat_cols] = oe.fit_transform(X[cat_cols])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Target — Train mean: {y_train.mean():,.2f}, Test mean: {y_test.mean():,.2f}")

## 16 · Preprocessing Strategy

- **Missing values**: None.
- **Categorical**: `weather`, `vehicle_type`, `city_traffic` encoded with OrdinalEncoder.
- **Scaling**: Not needed for tree models.
- **Outliers**: Long deliveries (>90 min) are genuine edge cases.

## 17 · Feature Engineering

Sanitize column names and add engineered features.

In [ ]:
X_train = X_train.copy()
X_test = X_test.copy()

X_train["prep_time_proxy"] = X_train["num_items"] * 2 + (5 - X_train["restaurant_rating"]) * 2
X_test["prep_time_proxy"] = X_test["num_items"] * 2 + (5 - X_test["restaurant_rating"]) * 2

X_train["hour_sin"] = np.sin(2 * np.pi * X_train["order_hour"] / 24)
X_test["hour_sin"] = np.sin(2 * np.pi * X_test["order_hour"] / 24)

X_train["hour_cos"] = np.cos(2 * np.pi * X_train["order_hour"] / 24)
X_test["hour_cos"] = np.cos(2 * np.pi * X_test["order_hour"] / 24)

clean = [c.replace(" ", "_").replace("-", "_").replace(".", "_") for c in X_train.columns]
X_train.columns = clean
X_test.columns = clean
print(f"Features ({len(clean)}): {clean}")

## 18 · Baseline Model

Linear Regression as our baseline regressor.

In [ ]:
baseline = LinearRegression()
baseline.fit(X_train, y_train)
y_pred_bl = baseline.predict(X_test)

rmse_bl = root_mean_squared_error(y_test, y_pred_bl)
mae_bl = mean_absolute_error(y_test, y_pred_bl)
r2_bl = r2_score(y_test, y_pred_bl)

print("=" * 50)
print("BASELINE: Linear Regression")
print("=" * 50)
print(f"RMSE : {rmse_bl:,.2f}")
print(f"MAE  : {mae_bl:,.2f}")
print(f"R2   : {r2_bl:.4f}")

## 19 · LazyPredict Benchmark

Fit dozens of regressors in one call for a quick ranking.

In [ ]:
from lazypredict.Supervised import LazyRegressor

lazy = LazyRegressor(verbose=0, ignore_warnings=True)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)

print("LazyPredict — Top 15 Regressors:")
print(lazy_models.head(15).to_string())

## 20 · FLAML AutoML Run

Automated model selection and hyperparameter tuning (60s budget).

In [ ]:
from flaml import AutoML

flaml_automl = AutoML()
flaml_automl.fit(X_train, y_train, task="regression", time_budget=60,
                 estimator_list=["lgbm", "rf", "extra_tree", "catboost"],
                 verbose=0, seed=SEED)
y_pred_flaml = flaml_automl.predict(X_test)
print(f"FLAML best model: {flaml_automl.best_estimator}")
print(f"RMSE: {root_mean_squared_error(y_test, y_pred_flaml):,.2f}")
print(f"R2  : {r2_score(y_test, y_pred_flaml):.4f}")

## 21 · Additional Models: CatBoost, LightGBM, XGBoost

Train the modern tabular model stack. GPU auto-detected with CPU fallback.

In [ ]:
def gpu_cleanup():
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass

results = {}
timings = {}

# CatBoost
try:
    from catboost import CatBoostRegressor
    t0 = time.perf_counter()
    try:
        cb = CatBoostRegressor(iterations=300, learning_rate=0.05, depth=6,
                                task_type="GPU", devices="0", verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
    except Exception:
        cb = CatBoostRegressor(iterations=300, learning_rate=0.05, depth=6,
                                verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
    timings["CatBoost"] = time.perf_counter() - t0
    results["CatBoost"] = cb.predict(X_test)
    print(f"CatBoost RMSE: {root_mean_squared_error(y_test, results['CatBoost']):,.2f}  ({timings['CatBoost']:.1f}s)")
except Exception as e:
    print(f"CatBoost failed: {e}")
gpu_cleanup()

# LightGBM
try:
    import lightgbm as lgb
    t0 = time.perf_counter()
    try:
        lg = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                                device="gpu", verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    except Exception:
        lg = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                                verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    timings["LightGBM"] = time.perf_counter() - t0
    results["LightGBM"] = lg.predict(X_test)
    print(f"LightGBM RMSE: {root_mean_squared_error(y_test, results['LightGBM']):,.2f}  ({timings['LightGBM']:.1f}s)")
except Exception as e:
    print(f"LightGBM failed: {e}")
gpu_cleanup()

# XGBoost
try:
    from xgboost import XGBRegressor
    t0 = time.perf_counter()
    try:
        xgb_model = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                                  device="cuda", tree_method="hist", verbosity=0,
                                  n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=30, verbose=False)
    except Exception:
        xgb_model = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                                  tree_method="hist", verbosity=0,
                                  n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=30, verbose=False)
    timings["XGBoost"] = time.perf_counter() - t0
    results["XGBoost"] = xgb_model.predict(X_test)
    print(f"XGBoost RMSE: {root_mean_squared_error(y_test, results['XGBoost']):,.2f}  ({timings['XGBoost']:.1f}s)")
except Exception as e:
    print(f"XGBoost failed: {e}")
gpu_cleanup()

# Add baseline & FLAML
results["Linear Regression"] = y_pred_bl
results["FLAML"] = y_pred_flaml

## 22 · Top 3 Model Selection

Rank all models by RMSE and select the top 3.

In [ ]:
model_scores = {}
for name, yp in results.items():
    model_scores[name] = {
        "RMSE": round(root_mean_squared_error(y_test, yp), 2),
        "MAE": round(mean_absolute_error(y_test, yp), 2),
        "R2": round(r2_score(y_test, yp), 4),
    }

scores_df = pd.DataFrame(model_scores).T.sort_values("RMSE")
print("All Model Rankings (by RMSE):")
print(scores_df.to_string())

top3_names = scores_df.index[:3].tolist()
print(f"\nTop 3 models: {top3_names}")

## 23 · Final Training and Evaluation of Top 3

Detailed metrics and predicted-vs-actual plots.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, name in enumerate(top3_names):
    yp = results[name]
    rmse = root_mean_squared_error(y_test, yp)
    r2 = r2_score(y_test, yp)

    axes[i].scatter(y_test, yp, alpha=0.6, s=40, edgecolors="black")
    mn, mx = min(y_test.min(), yp.min()), max(y_test.max(), yp.max())
    axes[i].plot([mn, mx], [mn, mx], "r--", lw=2)
    axes[i].set_title(f"{name}\nRMSE={rmse:,.2f}  R2={r2:.4f}")
    axes[i].set_xlabel("Actual")
    axes[i].set_ylabel("Predicted")

plt.suptitle("Top 3 Models — Predicted vs Actual", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "top3_predicted_vs_actual.png"), dpi=120, bbox_inches="tight")
plt.show()

print("\nDetailed Metrics — Top 3:")
for name in top3_names:
    yp = results[name]
    print(f"\n  {name}:")
    print(f"    RMSE : {root_mean_squared_error(y_test, yp):,.2f}")
    print(f"    MAE  : {mean_absolute_error(y_test, yp):,.2f}")
    print(f"    R2   : {r2_score(y_test, yp):.4f}")

## 24 · Error Analysis

Examine residuals from the best model.

In [ ]:
best_name = top3_names[0]
best_pred = results[best_name]
residuals = y_test.values - best_pred

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(residuals, bins=20, color="steelblue", edgecolor="black", alpha=0.8)
axes[0].axvline(0, color="red", linestyle="--")
axes[0].set_title(f"{best_name} — Residual Distribution")
axes[0].set_xlabel("Residual (Actual - Predicted)")

axes[1].scatter(best_pred, residuals, alpha=0.6, s=40, edgecolors="black")
axes[1].axhline(0, color="red", linestyle="--")
axes[1].set_title(f"{best_name} — Residual vs Predicted")
axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("Residual")

sorted_idx = np.argsort(best_pred)
axes[2].scatter(best_pred[sorted_idx], np.abs(residuals[sorted_idx]), alpha=0.6, s=40, edgecolors="black")
axes[2].set_title(f"{best_name} — |Residual| vs Predicted")
axes[2].set_xlabel("Predicted"); axes[2].set_ylabel("|Residual|")

plt.tight_layout()
plt.show()

print(f"Residual stats ({best_name}):")
print(f"  Mean: {residuals.mean():,.2f}, Std: {residuals.std():,.2f}")
print(f"  Min: {residuals.min():,.2f}, Max: {residuals.max():,.2f}")
print(f"  Median: {np.median(residuals):,.2f}")

## 25 · Interpretation and Business Insight

**Key findings:**
- **Distance** is the strongest delivery time predictor.
- **City traffic** level multiplies travel time significantly.
- **Weather** (Snow +12 min, Rain +5 min) adds predictable delays.
- **Peak hours** (lunch/dinner) add ~8 minutes.
- **Vehicle type** — motorcycles are fastest; bicycles slower for long distances.
- **Restaurant rating** inversely correlates with prep time.

**Business takeaway:** Assign motorcycles to long-distance/high-traffic orders. Pad ETA estimates during rain/snow and peak hours.

## 26 · Limitations

1. Synthetic data with simplified time model.
2. No real-time traffic data.
3. Missing driver experience factor.
4. No GPS-based actual route data.
5. Restaurant prep time is simplified.

## 27 · How to Improve This Project

1. Use real delivery platform data with GPS traces.
2. Add real-time traffic API integration.
3. Include driver experience and rating.
4. Model restaurant prep time separately.
5. Add order complexity features.

## 28 · Production Considerations

- Deploy for real-time ETA estimation.
- Show confidence intervals (e.g., 25-35 min).
- Monitor actual vs. predicted for model drift.
- Integrate with driver dispatch optimization.
- Update hourly for traffic pattern changes.

## 29 · Common Mistakes

1. Not decomposing delivery time (prep + travel + delay).
2. Ignoring weather impact on travel time.
3. Using average speed instead of traffic-adjusted speed.
4. Not accounting for peak-hour restaurant queue times.
5. Predicting without real-time traffic data.

## 30 · Mini Challenge / Exercises

1. Decompose predictions: prep time + travel time + delays.
2. Remove `city_traffic` — how much does MAE increase?
3. Plot delivery time vs. distance, colored by vehicle type.
4. Create `estimated_speed = distance / delivery_minutes` and analyze.
5. Build separate models for peak vs. off-peak hours.

## 31 · Final Summary / Key Takeaways

1. **Distance** is the dominant delivery time driver.
2. **Traffic** multiplies travel time (up to 1.4×).
3. **Weather** adds fixed delay components.
4. **Peak hours** add congestion and restaurant wait time.
5. **Vehicle type** matters most for long distances.
6. **ETA accuracy** directly drives customer satisfaction.

## Save Metrics

In [ ]:
metrics_out = {}
for name, yp in results.items():
    metrics_out[name] = {
        "rmse": round(float(root_mean_squared_error(y_test, yp)), 2),
        "mae": round(float(mean_absolute_error(y_test, yp)), 2),
        "r2": round(float(r2_score(y_test, yp)), 4),
    }

metrics_path = os.path.join(SAVE_DIR, "metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics_out, f, indent=2)
print(f"Metrics saved to {metrics_path}")
print(json.dumps(metrics_out, indent=2))